In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")


In [10]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:4b", base_url="http://192.168.101.78:11434")


In [11]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [12]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user's primary goal is to investigate a series of fictional details regarding Lunapolis (the capital of the moon), specifically focusing on its weather conditions, the population of cheese miners, and the likelihood of a labor strike.\n\n## SUMMARY\n- **Location:** Lunapolis is identified as the capital of the moon.\n- **Weather:** Clear skies, with a high temperature of 120C and a low of -100C.\n- **Demographics:** There are 100,000 cheese miners living in Lunapolis.\n- **Labor Unrest:** The cheese miners' union is likely to strike because they are unhappy with the new president.\n- **Rejected Options:** None were discussed or considered.\n\n## ARTIFACTS\nNone. No files were created, modified, or accessed during the session.\n\n## NEXT STEPS\nDetermine the specific reasons for the union's dissatisfaction and investigate any potential consequences of the planned strike.", additi

In [13]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to investigate a series of fictional details regarding Lunapolis (the capital of the moon), specifically focusing on its weather conditions, the population of cheese miners, and the likelihood of a labor strike.

## SUMMARY
- **Location:** Lunapolis is identified as the capital of the moon.
- **Weather:** Clear skies, with a high temperature of 120C and a low of -100C.
- **Demographics:** There are 100,000 cheese miners living in Lunapolis.
- **Labor Unrest:** The cheese miners' union is likely to strike because they are unhappy with the new president.
- **Rejected Options:** None were discussed or considered.

## ARTIFACTS
None. No files were created, modified, or accessed during the session.

## NEXT STEPS
Determine the specific reasons for the union's dissatisfaction and investigate any potential consequences of the planned strike.


## Trim/delete messages

In [14]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [15]:
agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [16]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='9c4b4b2a-b168-4b12-af18-0ae695ea2dcb'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='11c47d8e-3a66-415f-9de3-18e40d1cb84b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='f8dddcda-cf09-408d-a529-ccbdb26130d1'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='4495809f-9fe7-4352-b4c3-c91cb3b94b2b', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='14df32d4-f284-402e-8101-12f17b38cf41'),
              AIMessage(content='I am an AI and do not have access to the hardware of your devi

In [17]:
print(response["messages"][-1].content)

I am an AI and do not have access to the hardware of your device, so I cannot check its temperature directly. However, if it is overheating, it may have turned off to protect itself.

Can you feel if the device is very warm to the touch? Also, do you see any lights or indicator LEDs on it? Let me know, and I can guide you further.
